# Notebook 01 — Setup & Authentication

**RHOAI 3.5 | MLflow 3.13.0 | SDK minimum: `mlflow[kubernetes]>=3.11`**

This notebook covers:
1. Installing and verifying the MLflow SDK
2. Two authentication modes supported in RHOAI
3. Verifying connectivity to your MLflow workspace
4. Creating your first experiment

---

## How Authentication Works in RHOAI

Every MLflow API call is authorized by Kubernetes RBAC via `SelfSubjectAccessReview`. There is no MLflow-native user database — your OpenShift identity controls access.

| Mode | How it works | When to use |
|------|-------------|-------------|
| **Automatic (Workbench)** | `MLFLOW_TRACKING_AUTH=kubernetes-namespaced` injected by RHOAI | Running inside a workbench with the MLflow annotation |
| **Manual (Token)** | `MLFLOW_TRACKING_TOKEN` + `MLFLOW_WORKSPACE` | CI/CD, external scripts, dev without a workbench |

## Step 1 — Install the MLflow SDK

The `[kubernetes]` extra installs the RHOAI authentication plugin.

**In a RHOAI workbench the SDK is pre-installed** — this cell will print `Requirement already satisfied` and that is expected. Run it anyway to confirm the version constraint is met.

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

MIN_VERSION = "3.11.0"

try:
    installed = version("mlflow")
    if Version(installed) >= Version(MIN_VERSION):
        print(f"mlflow {installed} already installed — requirement >=3.11 met.")
    else:
        raise PackageNotFoundError(f"mlflow {installed} is below {MIN_VERSION}")
except PackageNotFoundError as e:
    print(f"ERROR: {e}")
    print()
    print("mlflow[kubernetes] is not available in this workbench image.")
    print("On a disconnected cluster, install via a custom workbench image.")
    print()
    print("Build steps (run on a connected machine, then push to your internal registry):")
    print("  1. Create a Containerfile — see 00_cluster_setup.ipynb, Section: Custom Workbench Image")
    print("  2. podman build -t <internal-registry>/mlflow-workbench:3.5 .")
    print("  3. podman push <internal-registry>/mlflow-workbench:3.5")
    print("  4. Register the image in RHOAI Dashboard > Settings > Notebook images")
    print()
    raise SystemExit("Notebook cannot continue without mlflow. Fix the workbench image first.")

In [ ]:
import mlflow
from packaging.version import Version

sdk_version = mlflow.__version__
minimum_version = "3.11.0"
server_version = "3.13.0"  # RHOAI 3.5 ships this version

print(f"MLflow SDK installed : {sdk_version}")
print(f"MLflow server (RHOAI): {server_version}")
print(f"Minimum required     : {minimum_version}")

if Version(sdk_version) < Version(minimum_version):
    raise RuntimeError(
        f"SDK {sdk_version} is below the minimum {minimum_version}. "
        "Run: pip install 'mlflow[kubernetes]>=3.11'"
    )
print("\nSDK version check passed.")

## Step 2 — Authentication

### Path A: Automatic (Workbench with MLflow annotation)

When your RHOAI workbench has the `opendatahub.io/mlflow-instance` annotation, these environment variables are **automatically injected** at notebook start:

```
MLFLOW_TRACKING_URI      = https://<route>/workspaces/<namespace>
MLFLOW_TRACKING_AUTH     = kubernetes-namespaced
MLFLOW_K8S_INTEGRATION   = true
```

A `RoleBinding` named `<notebook_name>-mlflow` is also created automatically, granting `mlflow-operator-mlflow-integration` ClusterRole to the notebook's service account.

### Path B: Manual Configuration

Use the cell below when running outside a pre-configured workbench.

In [ ]:
import os

# Check if RHOAI injected the env vars automatically
auto_uri  = os.environ.get("MLFLOW_TRACKING_URI")
auto_auth = os.environ.get("MLFLOW_TRACKING_AUTH")

if auto_uri and auto_auth == "kubernetes-namespaced":
    print("Automatic workbench integration detected.")
    print(f"  MLFLOW_TRACKING_URI  : {auto_uri}")
    print(f"  MLFLOW_TRACKING_AUTH : {auto_auth}")
    print(f"  K8S_INTEGRATION      : {os.environ.get('MLFLOW_K8S_INTEGRATION', 'not set')}")
else:
    print("Automatic integration NOT detected. Configuring manually...")
    print()

    # --- EDIT THESE VALUES ---
    MLFLOW_TRACKING_URI = "https://mlflow-<namespace>.<cluster-domain>/workspaces/<namespace>"
    AUTH_MODE = "kubernetes-namespaced"  # preferred; reads ~/.kube/config or service account token
    # AUTH_MODE = "bearer-token"         # alternative; set MLFLOW_TRACKING_TOKEN + MLFLOW_WORKSPACE
    # --------------------------

    os.environ["MLFLOW_TRACKING_URI"]  = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_TRACKING_AUTH"] = AUTH_MODE
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

    print(f"  Tracking URI set to  : {MLFLOW_TRACKING_URI}")
    print(f"  Auth mode            : {AUTH_MODE}")
    print()
    print("NOTE: Replace the placeholder URL above with your actual MLflow route.")
    print("      Find it at: RHOAI Dashboard > Data Science Projects > <project> > MLflow")

### Alternative: Token-Based Authentication

For CI/CD pipelines or environments where `kubeconfig` is not available, you can use a bearer token.
Obtain it with: `oc whoami --show-token`

In [ ]:
# OPTIONAL — uncomment and fill in if using token-based auth
# import os
#
# os.environ["MLFLOW_TRACKING_AUTH"]  = "bearer-token"
# os.environ["MLFLOW_TRACKING_TOKEN"] = "<your-oc-token>"   # oc whoami --show-token
# os.environ["MLFLOW_WORKSPACE"]      = "<your-namespace>"  # OpenShift project name
#
# NOTE: Do not use token auth in production — rotate tokens and use kubernetes-namespaced auth instead.
print("Token auth cell (commented out). Uncomment and fill in values if needed.")

## Step 3 — Optional: Direct S3 Transfer (Performance Tuning)

Enable multipart upload/download for large artifact transfers (models, datasets) to bypass the MLflow proxy server and write directly to S3.

In [ ]:
# Bypass the MLflow proxy and transfer artifacts directly to/from S3
os.environ["MLFLOW_ENABLE_PROXY_MULTIPART_UPLOAD"]   = "true"
os.environ["MLFLOW_ENABLE_PROXY_MULTIPART_DOWNLOAD"] = "true"

# Tuning knobs (adjust based on your network and artifact sizes)
os.environ["MLFLOW_MULTIPART_UPLOAD_MINIMUM_FILE_SIZE"] = "10485760"  # 10 MB — files smaller than this use single-part upload
os.environ["MLFLOW_MULTIPART_UPLOAD_CHUNK_SIZE"]        = "10485760"  # 10 MB chunks
os.environ["MLFLOW_MULTIPART_DOWNLOAD_CHUNK_SIZE"]      = "10485760"  # 10 MB chunks

print("Direct S3 multipart transfer enabled.")
print("  Minimum file size for multipart: 10 MB")
print("  Upload chunk size               : 10 MB")
print("  Download chunk size             : 10 MB")

## Step 4 — Verify Connectivity

In [ ]:
import mlflow

print("Attempting to list workspaces...")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

try:
    workspaces = mlflow.list_workspaces()
    print(f"\nConnection successful. {len(workspaces)} workspace(s) accessible:")
    for ws in workspaces:
        print(f"  - {ws.name}")
except Exception as e:
    print(f"\nConnection failed: {e}")
    print()
    print("Troubleshooting checklist:")
    print("  1. Is your MLFLOW_TRACKING_URI correct?")
    print("  2. Do you have at least mlflow-view ClusterRole in this namespace?")
    print("  3. Is the MLflow workspace deployed? Check: oc get mlflowtrackingserver -n <namespace>")
    print("  4. Is the MLflow Operator enabled? Check DataScienceCluster for mlflowoperator: Managed")

In [ ]:
# Display the tracking server version
import mlflow.tracking

client = mlflow.tracking.MlflowClient()

try:
    # List existing experiments to confirm the server is responding
    experiments = client.search_experiments()
    print(f"Server is responding. {len(experiments)} experiment(s) already exist:")
    for exp in experiments:
        print(f"  [{exp.experiment_id}] {exp.name}  (lifecycle: {exp.lifecycle_stage})")
except Exception as e:
    print(f"Error reaching server: {e}")

## Step 5 — Create Your First Experiment

In RHOAI, experiments are scoped to your workspace (namespace). The `Default` experiment always exists.

In [ ]:
EXPERIMENT_NAME = "rhoai-mlflow-demo"

# create_experiment raises if it already exists; get_or_create_experiment is idempotent
exp = mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment name  : {exp.name}")
print(f"Experiment ID    : {exp.experiment_id}")
print(f"Artifact location: {exp.artifact_location}")
print(f"Lifecycle stage  : {exp.lifecycle_stage}")

## Step 6 — Hello World Run

A minimal run to confirm the full tracking pipeline works end-to-end.

In [ ]:
import mlflow

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="hello-world") as run:
    # Log a parameter (hyperparameter)
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("framework", "demo")

    # Log metrics over 3 epochs
    for epoch in range(1, 4):
        mlflow.log_metric("loss", 1.0 / epoch, step=epoch)
        mlflow.log_metric("accuracy", 0.5 + epoch * 0.15, step=epoch)

    # Log a tag
    mlflow.set_tag("demo_notebook", "01_setup")
    mlflow.set_tag("rhoai_version", "3.5")

    run_id = run.info.run_id

print(f"Run ID     : {run_id}")
print(f"Run status : {run.info.status}")
print()
print("View this run in the RHOAI Dashboard:")
print("  Develop & train > Experiments (MLflow) > rhoai-mlflow-demo")

## Summary

| Task | Status |
|------|--------|
| SDK installed (>=3.11) | ✓ |
| Authentication configured | ✓ |
| Workspace reachable | ✓ |
| Experiment created | ✓ |
| Hello-world run logged | ✓ |

**Next:** Proceed to `02_experiment_tracking.ipynb` for full experiment tracking with a real ML model.